In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
y_train = y_train.ravel()
y_test = y_test.ravel()

IMG_SIZE = (224, 224)   # مناسب لـ VGG & ResNet
BATCH = 64

# Resize + Normalize
def preprocess(x, y, train=False):
    x = tf.image.resize(x, IMG_SIZE)
    x = tf.cast(x, tf.float32)/255.0
    if train:
        x = tf.image.random_flip_left_right(x)
    return x, y

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(20000)
    .map(lambda x,y: preprocess(x,y,True))
    .batch(BATCH).prefetch(2)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(lambda x,y: preprocess(x,y,False))
    .batch(BATCH).prefetch(2)
)


In [ ]:
def build_scratch():
    inputs = layers.Input(shape=(224,224,3))
    x = layers.Conv2D(32, 3, activation='relu')(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation='relu')(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    outputs = layers.Dense(10, activation='softmax')(x)
    return models.Model(inputs, outputs)

model_scratch = build_scratch()
model_scratch.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model_scratch.fit(train_ds, epochs=3, validation_data=test_ds)
model_scratch.save("cnn_scratch.h5")


In [ ]:
base = tf.keras.applications.ResNet50(
    include_top=False, weights="imagenet", input_shape=(224,224,3), pooling="avg"
)


In [ ]:
base.trainable = False   # تجميد الموديل

inputs = layers.Input(shape=(224,224,3))
x = base(inputs, training=False)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(10, activation='softmax')(x)
model_tl = models.Model(inputs, outputs)

model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_tl.fit(train_ds, epochs=3, validation_data=test_ds)
model_tl.save("model_frozen.h5")


In [ ]:
# نفك آخر 30% من الليرات
base.trainable = True

fine_tune_at = int(len(base.layers) * 0.7)

for i, layer in enumerate(base.layers):
    layer.trainable = i >= fine_tune_at

model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # LR صغير جداً
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_tl.fit(train_ds, epochs=5, validation_data=test_ds)
model_tl.save("model_finetuned.h5")
